# Predicción de adopción de perros y gatos – Austin Animal Center
Curso Inteligencia Artificial (1INF24) – Tarea Académica, entregable final.
Variables de entrada: **Intakes** (estado del animal al ingresar). Variable objetivo: **Outcomes** (1 = Adoption).

In [ ]:
import re, time, json, warnings, os
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score, roc_curve,
                             confusion_matrix, accuracy_score)
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")

# ---- PARÁMETROS (cambiar rutas si hace falta) ----
INTAKES  = "Austin_Animal_Center_Intakes__10_01_2013_to_05_05_2025_.csv"
OUTCOMES = "Austin_Animal_Center_Outcomes__10_01_2013_to_05_05_2025_.csv"
OUT_DIR  = "resultados"
SEED = 42
TUNE_SAMPLE = None   # None = ajustar con todo el entrenamiento; p.ej. 40000 para ir más rápido
os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)

## 1. Construcción del conjunto de datos (Intake + Outcome)

In [ ]:
def parse_age_days(s):
    if not isinstance(s, str): return np.nan
    m = re.match(r'\s*(-?\d+)\s*(day|week|month|year)', s)
    if not m: return np.nan
    n, u = int(m.group(1)), m.group(2)
    return np.nan if n < 0 else n * {'day':1,'week':7,'month':30,'year':365}[u]

def build_dataset(intakes_csv, outcomes_csv):
    i = pd.read_csv(intakes_csv); o = pd.read_csv(outcomes_csv)
    i['in_date']  = pd.to_datetime(i['DateTime'], format='%m/%d/%Y %I:%M:%S %p').dt.normalize()
    o['out_date'] = pd.to_datetime(o['DateTime'].str[:10], format='%Y-%m-%d')
    i = i[i['Animal Type'].isin(['Dog','Cat'])].copy()
    o = o[o['Animal Type'].isin(['Dog','Cat'])].copy()
    stats = {'ingresos_perros_gatos': len(i)}
    # cada ingreso se empareja con la primera salida posterior del mismo animal
    i = i.sort_values('in_date'); o = o.sort_values('out_date')
    m = pd.merge_asof(i, o[['Animal ID','out_date','Outcome Type']], left_on='in_date',
                      right_on='out_date', by='Animal ID', direction='forward')
    stats['sin_salida_registrada'] = int(m['out_date'].isna().sum())
    m = m[m['out_date'].notna()].sort_values('in_date')
    n0 = len(m); m = m.drop_duplicates(['Animal ID','out_date'], keep='last')
    stats['ingresos_repetidos_misma_salida'] = n0 - len(m)
    m = m[m['Outcome Type'].notna()].copy()
    stats['registros_finales'] = len(m); stats['animales_distintos'] = int(m['Animal ID'].nunique())
    m['adopted'] = (m['Outcome Type'] == 'Adoption').astype(int)
    # variables conocidas AL INGRESO
    sx = m['Sex upon Intake'].fillna('Unknown')
    m['sexo'] = np.where(sx.str.contains('Female'), 'Hembra', np.where(sx.str.contains('Male'), 'Macho', 'Desconocido'))
    m['cond_reproductiva'] = np.where(sx.str.contains('Intact'), 'Entero',
                              np.where(sx.str.contains('Neutered|Spayed'), 'Esterilizado', 'Desconocido'))
    m['edad_dias'] = m['Age upon Intake'].map(parse_age_days)
    m['edad_dias'] = m['edad_dias'].fillna(m['edad_dias'].median())
    m['es_mestizo'] = m['Breed'].str.contains('Mix|/', case=False, regex=True).astype(int)
    m['raza_principal'] = m['Breed'].str.replace(' Mix', '', regex=False).str.split('/').str[0].str.strip()
    m['color_principal'] = m['Color'].str.split('/').str[0].str.strip()
    m = m.rename(columns={'Animal Type':'tipo_animal','Intake Type':'tipo_ingreso','Intake Condition':'condicion_ingreso'})
    return m.reset_index(drop=True), stats

df, stats = build_dataset(INTAKES, OUTCOMES)
print(stats); print("Tasa de adopción:", round(df.adopted.mean(), 4))
json.dump(stats, open(f"{OUT_DIR}/estadisticas_datos.json", "w"), indent=2)
df['Outcome Type'].value_counts().to_csv(f"{OUT_DIR}/tabla_salidas.csv")

CAT = ['tipo_animal','tipo_ingreso','condicion_ingreso','sexo','cond_reproductiva','raza_principal','color_principal']
NUM = ['edad_dias','es_mestizo']
X, y, groups = df[CAT+NUM], df['adopted'], df['Animal ID']

## 2. División 80/20 estratificada, sin repetir animales entre entrenamiento y prueba

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx, te_idx = next(sgkf.split(X, y, groups))
Xtr, Xte, ytr, yte = X.iloc[tr_idx], X.iloc[te_idx], y.iloc[tr_idx], y.iloc[te_idx]
assert set(groups.iloc[tr_idx]).isdisjoint(set(groups.iloc[te_idx]))
split_info = dict(n_train=len(Xtr), n_test=len(Xte), adopt_train=float(ytr.mean()), adopt_test=float(yte.mean()))
print(split_info); json.dump(split_info, open(f"{OUT_DIR}/division.json","w"), indent=2)

def make_pre(scale=False):
    # categorías con <1 % de frecuencia se agrupan automáticamente en "otras" (infrequent)
    ohe = OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=0.01, sparse_output=False)
    num = StandardScaler() if scale else 'passthrough'
    return ColumnTransformer([('cat', ohe, CAT), ('num', num, NUM)])

## 3. Modelos, búsqueda en malla y validación cruzada de 5 pliegues (solo con entrenamiento)

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
models = {
 'Línea base (mayoritaria)': (DummyClassifier(strategy='most_frequent'), {}, False),
 'Regresión logística': (LogisticRegression(max_iter=2000), {'clf__C':[0.01,0.1,1,10]}, True),
 'Árbol de decisión': (DecisionTreeClassifier(random_state=SEED),
        {'clf__max_depth':[5,10,15,20], 'clf__min_samples_leaf':[10,50,100]}, False),
 'Random Forest': (RandomForestClassifier(random_state=SEED, n_jobs=-1),
        {'clf__n_estimators':[100,200], 'clf__max_depth':[10,20]}, False),
 'Gradient Boosting': (HistGradientBoostingClassifier(random_state=SEED),
        {'clf__learning_rate':[0.05,0.1,0.2], 'clf__max_iter':[100,200], 'clf__max_depth':[3,6]}, False),
}
if TUNE_SAMPLE and TUNE_SAMPLE < len(Xtr):
    sub = np.random.RandomState(SEED).choice(len(Xtr), TUNE_SAMPLE, replace=False)
else:
    sub = np.arange(len(Xtr))

fitted, rows, best_params = {}, [], {}
for name, (clf, grid, scale) in models.items():
    t0 = time.time()
    pipe = Pipeline([('pre', make_pre(scale)), ('clf', clf)])
    if grid:
        gs = GridSearchCV(pipe, grid, scoring='f1', cv=cv, n_jobs=1, refit=False)
        gs.fit(Xtr.iloc[sub], ytr.iloc[sub], groups=groups.iloc[tr_idx].iloc[sub])
        best_params[name] = {k.replace('clf__',''): (v.item() if hasattr(v,'item') else v) for k, v in gs.best_params_.items()}
        cv_f1 = float(gs.best_score_)
        pipe.set_params(**gs.best_params_)
    else:
        cv_f1 = np.nan
    pipe.fit(Xtr, ytr)                      # reajuste final con todo el entrenamiento
    fitted[name] = pipe
    print(f"{name}: {best_params.get(name,'-')}  CV-F1={cv_f1:.4f}  ({time.time()-t0:.0f}s)", flush=True)
    rows.append(dict(modelo=name, cv_f1=cv_f1, tiempo_s=round(time.time()-t0,1)))
json.dump(best_params, open(f"{OUT_DIR}/mejores_hiperparametros.json","w"), indent=2, ensure_ascii=False)
times = pd.DataFrame(rows).set_index('modelo')

## 4. Evaluación en el conjunto de prueba (por clase)

In [ ]:
res, probs = [], {}
for name, pipe in fitted.items():
    p = pipe.predict_proba(Xte)[:, 1]; pred = (p >= 0.5).astype(int); probs[name] = p
    pr, rc, f1, _ = precision_recall_fscore_support(yte, pred, labels=[0,1], zero_division=0)
    res.append({'Modelo': name, 'Exactitud': accuracy_score(yte, pred),
                'Precisión (0)': pr[0], 'Recall (0)': rc[0], 'F1 (0)': f1[0],
                'Precisión (1)': pr[1], 'Recall (1)': rc[1], 'F1 (1)': f1[1],
                'F1 macro': (f1[0]+f1[1])/2, 'AUC-ROC': roc_auc_score(yte, p)})
tabla = pd.DataFrame(res).set_index('Modelo').round(4)
tabla['F1 CV (entren.)'] = times['cv_f1'].round(4)
tabla.to_csv(f"{OUT_DIR}/tabla_resultados.csv"); print(tabla.T)

## 5. Figuras

In [ ]:
plt.rcParams.update({'font.size': 9})
# Fig A: distribución de salidas
vc = df['Outcome Type'].value_counts()
fig, ax = plt.subplots(figsize=(5,3)); vc.head(6).plot.barh(ax=ax, color='#1b7f5c'); ax.invert_yaxis()
ax.set_xlabel('Número de registros'); ax.set_title('Salidas registradas (perros y gatos)')
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_salidas.png", dpi=200); plt.close()

# Fig B: tasa de adopción por edad y por tipo de ingreso
df['grupo_edad'] = pd.cut(df['edad_dias'], [-1,60,365,1095,2555,99999], labels=['<2 m','2 m–1 a','1–3 a','3–7 a','>7 a'])
fig, axs = plt.subplots(1,2, figsize=(8,3))
df.groupby('grupo_edad')['adopted'].mean().plot.bar(ax=axs[0], color='#1b7f5c'); axs[0].set_ylabel('Tasa de adopción'); axs[0].set_xlabel('Edad al ingreso'); axs[0].tick_params(axis='x', rotation=0)
df.groupby('tipo_ingreso')['adopted'].mean().sort_values().plot.barh(ax=axs[1], color='#1b7f5c'); axs[1].set_xlabel('Tasa de adopción'); axs[1].set_ylabel('')
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_tasa_adopcion.png", dpi=200); plt.close()

# Fig C: curvas ROC
fig, ax = plt.subplots(figsize=(4.2,3.8))
for name, p in probs.items():
    if name.startswith('Línea'): continue
    fpr, tpr, _ = roc_curve(yte, p); ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(yte,p):.3f})")
ax.plot([0,1],[0,1],'k--',lw=.8,label='Azar'); ax.set_xlabel('Tasa de falsos positivos'); ax.set_ylabel('Tasa de verdaderos positivos')
ax.set_title('Curvas ROC (conjunto de prueba)'); ax.legend(fontsize=7, loc='lower right')
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_roc.png", dpi=200); plt.close()

# Fig D: matrices de confusión
names = [n for n in fitted if not n.startswith('Línea')]
fig, axs = plt.subplots(1,4, figsize=(11,2.8))
for ax, n in zip(axs, names):
    cm = confusion_matrix(yte, (probs[n]>=.5).astype(int)); ax.imshow(cm, cmap='Greens')
    for a in range(2):
        for b in range(2): ax.text(b, a, f"{cm[a,b]:,}", ha='center', va='center', fontsize=8)
    ax.set_xticks([0,1]); ax.set_yticks([0,1]); ax.set_xticklabels(['No adopt.','Adoptado']); ax.set_yticklabels(['No adopt.','Adoptado'])
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real'); ax.set_title(n, fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_confusion.png", dpi=200); plt.close()

# Fig E: importancia por permutación (variable original) del mejor modelo según AUC
best = tabla.drop(index='Línea base (mayoritaria)')['AUC-ROC'].idxmax(); print("Mejor modelo (AUC):", best)
ss = np.random.RandomState(SEED).choice(len(Xte), min(15000, len(Xte)), replace=False)
pi = permutation_importance(fitted[best], Xte.iloc[ss], yte.iloc[ss], scoring='roc_auc', n_repeats=3, random_state=SEED, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=Xte.columns).sort_values()
imp.to_csv(f"{OUT_DIR}/importancia_variables.csv")
fig, ax = plt.subplots(figsize=(5,3)); imp.plot.barh(ax=ax, color='#1b7f5c'); ax.set_xlabel('Caída del AUC-ROC al permutar la variable')
ax.set_title(f'Importancia de variables – {best}'); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_importancia.png", dpi=200); plt.close()

## 6. Desempeño por subgrupo (implicancias éticas)

In [ ]:
te = df.iloc[te_idx].copy(); te['p'] = probs[best]; te['pred'] = (te['p']>=.5).astype(int)
def grp(g):
    return pd.Series({'n': len(g), 'tasa_adopción_real': g.adopted.mean(),
                      'AUC': roc_auc_score(g.adopted, g.p) if g.adopted.nunique()>1 else np.nan,
                      'recall_adoptados': ((g.pred==1)&(g.adopted==1)).sum()/max((g.adopted==1).sum(),1),
                      'recall_no_adoptados': ((g.pred==0)&(g.adopted==0)).sum()/max((g.adopted==0).sum(),1)})
sub_tipo = te.groupby('tipo_animal').apply(grp)
te['grupo_edad'] = pd.cut(te['edad_dias'], [-1,60,365,1095,2555,99999], labels=['<2 m','2 m–1 a','1–3 a','3–7 a','>7 a'])
sub_edad = te.groupby('grupo_edad').apply(grp)
pd.concat({'tipo':sub_tipo,'edad':sub_edad}).round(3).to_csv(f"{OUT_DIR}/tabla_subgrupos.csv")
print(pd.concat({'tipo':sub_tipo,'edad':sub_edad}).round(3))